# Notebook 04 — The ARAF Model
## Adaptive Reliability-Aware Fusion (ARAF) Project

**Goal of this notebook:**  
Build the core ARAF model — the actual research contribution of this project.
ARAF adds two components on top of the baselines from Notebook 03:
a Reliability Estimator and an Adaptive Fusion layer.

**What makes ARAF different from Naive Fusion:**  
Naive Fusion blindly concatenates image and text features regardless of quality.
ARAF first estimates how reliable each modality is, then weights the fusion
accordingly. A corrupted image gets a low reliability score and contributes
less to the final representation. A clean image gets a high score and
contributes more.

**What you will learn:**
- How to design a reliability estimator (what it looks at, what it outputs)
- How attention-based weighted fusion works
- How to add a regularization loss that trains the reliability estimator
- How the full ARAF forward pass differs from naive fusion
- How to visualize reliability scores to interpret what the model learned

---


## 1. Imports and setup


In [ ]:
import os, sys, json, copy, random
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from transformers import BertModel, BertTokenizer
from datasets import load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMAGE_SIZE    = 224
MAX_TEXT_LEN  = 32

clean_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img  = (tensor * std + mean).clamp(0, 1)
    return (img * 255).byte().permute(1, 2, 0).numpy()

print("Setup complete.")


## 2. Reload shared components

We import the baselines from the `.py` file saved in Notebook 03,
and reload the corruption module from Notebook 02.
This is what the workflow will look like going forward — notebooks
stay thin, logic lives in `.py` files.


In [ ]:
# ── MultimodalSample ─────────────────────────────────────────────────────────
@dataclass
class MultimodalSample:
    image: torch.Tensor
    text_ids: torch.Tensor
    attention_mask: torch.Tensor
    label: torch.Tensor
    raw_image: Optional[object] = None
    raw_text: str = ""
    dataset_name: str = "vqa_v2"
    sample_id: str = ""
    image_corrupted: bool = False
    text_corrupted: bool = False
    image_missing: bool = False
    text_missing: bool = False
    corruption_severity: float = 0.0

# ── Import baselines ──────────────────────────────────────────────────────────
import importlib.util

spec = importlib.util.spec_from_file_location("baselines", "models/baselines.py")
baselines = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baselines)

ImageEncoder      = baselines.ImageEncoder
TextEncoder       = baselines.TextEncoder
ClassificationHead= baselines.ClassificationHead
vqa_loss          = baselines.vqa_loss
vqa_accuracy      = baselines.vqa_accuracy
print("Baselines imported from models/baselines.py")

# ── Import corruption module ──────────────────────────────────────────────────
spec2 = importlib.util.spec_from_file_location(
    "corruption_module", "corruption/corruption_module.py")
corruption_mod = importlib.util.module_from_spec(spec2)
spec2.loader.exec_module(corruption_mod)
CorruptionModule = corruption_mod.CorruptionModule
print("CorruptionModule imported from corruption/corruption_module.py")


In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
print("Loading VQA v2 samples...")
hf_val_full = load_dataset("lmms-lab/VQAv2", split="validation",
                           trust_remote_code=True)
with open("answer_vocab.json") as f:
    answer2idx = json.load(f)
idx2answer   = {v: k for k, v in answer2idx.items()}
NUM_CLASSES  = len(answer2idx)
tokenizer    = BertTokenizer.from_pretrained("bert-base-uncased")

def load_sample(row):
    pil = row["image"].convert("RGB")
    img = clean_transform(pil)
    enc = tokenizer(row["question"], padding="max_length",
                    max_length=MAX_TEXT_LEN, truncation=True,
                    return_tensors="pt")
    label = torch.zeros(NUM_CLASSES)
    cnt = Counter(a["answer"].lower().strip() for a in row["answers"])
    for ans, c in cnt.items():
        if ans in answer2idx:
            label[answer2idx[ans]] = min(c / 3.0, 1.0)
    return MultimodalSample(
        image=img, text_ids=enc["input_ids"].squeeze(0),
        attention_mask=enc["attention_mask"].squeeze(0),
        label=label, raw_image=pil, raw_text=row["question"],
        dataset_name="vqa_v2",
        sample_id=str(row.get("question_id", 0)),
    )

samples = [load_sample(hf_val_full[i]) for i in range(64)]
print(f"Loaded {len(samples)} samples.")


## 3. The Reliability Estimator — concept before code

This is the core novelty of ARAF. Before writing any code, understand
what this component does and why it is designed the way it is.

### What problem it solves

When an image is corrupted with Gaussian noise, the ResNet encoder still
produces a 2048-dimensional feature vector — but those features are now
less meaningful. The question is: **how can the model know the features
are unreliable without being explicitly told?**

The answer is that corrupted features have different statistical properties
than clean features:
- Their L2 norm tends to be different (noise changes the magnitude)
- Their distribution shifts (corruption moves features off the clean manifold)
- Zero-filled inputs (missing modality) produce very distinctive near-zero features

The reliability estimator learns to detect these patterns.

### What it takes as input

The reliability estimator takes the **encoded feature vector** as input,
not the raw image or text. This is important because:
1. It operates in feature space, which is lower-dimensional and more structured
2. It's modality-agnostic — same design works for image and text features
3. It sees what the fusion module will see — if features are bad, it knows

### What it outputs

A single scalar value in [0, 1] per modality per sample:
- 0.0 = completely unreliable (missing or severely corrupted)
- 1.0 = fully reliable (clean input)

These scores become the weights in the adaptive fusion step.

### How it is trained

The reliability estimator has no direct supervision signal — we don't have
ground truth reliability labels. Instead it is trained via two indirect signals:

1. **Task loss backpropagation**: if the estimator up-weights a bad modality,
   the fusion produces worse predictions, and the loss penalizes this.
   Over time the estimator learns to down-weight bad modalities.

2. **Regularization loss**: we add an auxiliary loss that encourages the
   estimator to output low scores when we know the input is corrupted
   (we know this during training because we applied the corruption ourselves).
   This gives the estimator a direct training signal.


## 4. Building the Reliability Estimator


In [ ]:
class ReliabilityEstimator(nn.Module):
    """
    Estimates how reliable a modality's feature vector is.
    Takes encoded features as input, outputs a scalar reliability score in [0,1].

    Architecture:
        features [feature_dim]
            -> Linear(feature_dim, hidden_dim) -> ReLU -> Dropout
            -> Linear(hidden_dim, hidden_dim//2) -> ReLU
            -> Linear(hidden_dim//2, 1)
            -> Sigmoid   <- squashes to [0, 1]

    Why this architecture?
        Small and fast — reliability estimation should not dominate compute.
        Two hidden layers give enough capacity to detect corruption patterns.
        Sigmoid output maps naturally to a probability/weight interpretation.

    Args:
        feature_dim: dimension of input features (2048 for image, 768 for text)
        hidden_dim : hidden layer size (default 256, intentionally small)
        dropout    : dropout rate
    """
    def __init__(self, feature_dim: int, hidden_dim: int = 256,
                 dropout: float = 0.2):
        super().__init__()
        self.estimator = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid(),   # output in [0, 1]
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            features: [B, feature_dim]
        Returns:
            reliability: [B, 1]  values in [0, 1]
        """
        return self.estimator(features)


# ── Test ──────────────────────────────────────────────────────────────────────
print("Testing ReliabilityEstimator...")

# Image reliability estimator (feature_dim=2048)
img_reliability = ReliabilityEstimator(feature_dim=2048).to(DEVICE)
# Text reliability estimator (feature_dim=768)
txt_reliability = ReliabilityEstimator(feature_dim=768).to(DEVICE)

# Dummy features
dummy_img_feat = torch.randn(4, 2048).to(DEVICE)
dummy_txt_feat = torch.randn(4, 768).to(DEVICE)

with torch.no_grad():
    img_scores = img_reliability(dummy_img_feat)
    txt_scores = txt_reliability(dummy_txt_feat)

print(f"Image reliability scores shape : {img_scores.shape}")
print(f"Text  reliability scores shape : {txt_scores.shape}")
print(f"Image scores (random init)     : {img_scores.squeeze().tolist()}")
print(f"  (random init gives ~0.5 — after training these will diverge)")

trainable = sum(p.numel() for p in img_reliability.parameters())
print(f"Trainable params per estimator : {trainable:,}  (very small by design)")


## 5. Adaptive Fusion — weighted combination

Once we have reliability scores for each modality, we use them to
weight the contribution of each modality's features in the fusion.

### The fusion formula

Given:
- Image features `f_img` of shape `[B, 2048]`
- Text features `f_txt` of shape `[B, 768]`
- Image reliability score `r_img` of shape `[B, 1]`
- Text reliability score `r_txt` of shape `[B, 1]`

We first project both feature vectors to the same dimension `d` (512),
then compute a weighted sum:

```
f_img_proj = Linear(2048 -> d)   # [B, d]
f_txt_proj = Linear(768  -> d)   # [B, d]

fused = r_img * f_img_proj + r_txt * f_txt_proj   # [B, d]
```

### Why project to the same dimension first?

We cannot directly add a 2048-dim vector and a 768-dim vector — dimensions
must match for element-wise addition. The projection layers also give the
model freedom to re-represent each modality's features in a shared space
before combining them.

### Why weighted sum instead of weighted concatenation?

Weighted concatenation `[r_img * f_img, r_txt * f_txt]` would still pass
both vectors to the classifier regardless of reliability. A near-zero
reliability score would just zero out half the input — the classifier
would still see the zeros and might learn spurious patterns from them.

Weighted sum collapses to a single `[B, d]` vector. If image reliability
is 0.0, the image contributes nothing at all. If it is 1.0, image and
text contribute equally. The classifier sees one clean combined signal.

### Normalization option

We optionally normalize the weights so they sum to 1:
```
r_img_norm = r_img / (r_img + r_txt + eps)
r_txt_norm = r_txt / (r_img + r_txt + eps)
```
This ensures the scale of the fused features doesn't vary wildly
depending on whether one or both modalities are present.


In [ ]:
class AdaptiveFusion(nn.Module):
    """
    Reliability-weighted fusion of image and text features.

    Projects both modalities to a common dimension, then computes
    a reliability-weighted sum.

    Args:
        img_dim     : image feature dimension (2048)
        txt_dim     : text feature dimension (768)
        fusion_dim  : common projection dimension (default 512)
        normalize_weights: if True, normalize reliability scores to sum to 1
    """
    def __init__(self, img_dim: int = 2048, txt_dim: int = 768,
                 fusion_dim: int = 512, normalize_weights: bool = True):
        super().__init__()
        self.fusion_dim        = fusion_dim
        self.normalize_weights = normalize_weights

        # Project each modality to fusion_dim
        self.img_proj = nn.Sequential(
            nn.Linear(img_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.ReLU(),
        )
        self.txt_proj = nn.Sequential(
            nn.Linear(txt_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.ReLU(),
        )

    def forward(self, img_features: torch.Tensor,
                txt_features: torch.Tensor,
                img_reliability: torch.Tensor,
                txt_reliability: torch.Tensor) -> Dict:
        """
        Args:
            img_features    : [B, img_dim]
            txt_features    : [B, txt_dim]
            img_reliability : [B, 1]  scores in [0,1]
            txt_reliability : [B, 1]  scores in [0,1]
        Returns:
            dict with:
                fused          : [B, fusion_dim]  the weighted combination
                img_weight     : [B, 1]  effective image weight
                txt_weight     : [B, 1]  effective text weight
                img_proj       : [B, fusion_dim]  projected image features
                txt_proj       : [B, fusion_dim]  projected text features
        """
        # Project to common space
        img_p = self.img_proj(img_features)   # [B, fusion_dim]
        txt_p = self.txt_proj(txt_features)   # [B, fusion_dim]

        # Optionally normalize weights to sum to 1
        if self.normalize_weights:
            eps   = 1e-8
            total = img_reliability + txt_reliability + eps
            img_w = img_reliability / total   # [B, 1]
            txt_w = txt_reliability / total   # [B, 1]
        else:
            img_w = img_reliability
            txt_w = txt_reliability

        # Weighted sum: broadcast [B,1] across [B, fusion_dim]
        fused = img_w * img_p + txt_w * txt_p   # [B, fusion_dim]

        return {
            "fused"    : fused,
            "img_weight": img_w,
            "txt_weight": txt_w,
            "img_proj" : img_p,
            "txt_proj" : txt_p,
        }


# ── Test ──────────────────────────────────────────────────────────────────────
print("Testing AdaptiveFusion...")
fusion_layer = AdaptiveFusion(img_dim=2048, txt_dim=768,
                              fusion_dim=512).to(DEVICE)

dummy_img_feat = torch.randn(4, 2048).to(DEVICE)
dummy_txt_feat = torch.randn(4, 768).to(DEVICE)
dummy_img_rel  = torch.tensor([[0.9], [0.1], [0.5], [0.0]]).to(DEVICE)
dummy_txt_rel  = torch.tensor([[0.1], [0.9], [0.5], [1.0]]).to(DEVICE)

with torch.no_grad():
    fusion_out = fusion_layer(dummy_img_feat, dummy_txt_feat,
                              dummy_img_rel, dummy_txt_rel)

print(f"Fused features shape : {fusion_out['fused'].shape}")
print(f"Image weights        : {fusion_out['img_weight'].squeeze().tolist()}")
print(f"Text  weights        : {fusion_out['txt_weight'].squeeze().tolist()}")
print(f"Weight sums          : {(fusion_out['img_weight'] + fusion_out['txt_weight']).squeeze().tolist()}")
print(f"  (should all be ~1.0 with normalize_weights=True)")
print()
print("Manual check:")
print(f"  Sample 0: img_rel=0.9, txt_rel=0.1 -> img dominates (weight={fusion_out['img_weight'][0].item():.2f})")
print(f"  Sample 1: img_rel=0.1, txt_rel=0.9 -> txt dominates (weight={fusion_out['txt_weight'][1].item():.2f})")
print(f"  Sample 3: img_rel=0.0, txt_rel=1.0 -> txt only      (img_w={fusion_out['img_weight'][3].item():.2f})")


## 6. Reliability Regularization Loss

This is the auxiliary training signal that teaches the reliability
estimator what corrupted inputs look like.

### The problem without regularization

Without any direct supervision, the reliability estimator could learn
to output 0.5 for everything and still minimize the task loss reasonably
well (by letting the classifier compensate). We need to give it
a direct signal: "when this modality is corrupted, output a low score."

### The regularization loss

During training, we know which samples were corrupted because we applied
the corruption ourselves. We use this knowledge to compute:

```
target_score = 1.0  if modality is clean
target_score = 0.0  if modality is missing
target_score = 1.0 - corruption_severity  if modality is corrupted

reg_loss = MSE(predicted_score, target_score)
```

This is mean squared error between the estimator's output and what we
know the correct score should be. Over training, the estimator learns
to detect corruption from features alone — so at test time (when we
don't apply corruption and don't have the flag), it still works correctly.

### Total loss

```
total_loss = task_loss + lambda_reg * reg_loss
```

`lambda_reg` controls how strongly we enforce the reliability signal
vs. the task signal. We default to 0.1 — the task loss should dominate,
with the regularization as a guide.


In [ ]:
def reliability_regularization_loss(
    img_reliability: torch.Tensor,
    txt_reliability: torch.Tensor,
    batch: Dict,
    lambda_reg: float = 0.1,
) -> torch.Tensor:
    """
    Auxiliary loss that trains the reliability estimator using
    corruption metadata available during training.

    Args:
        img_reliability : [B, 1] predicted image reliability scores
        txt_reliability : [B, 1] predicted text reliability scores
        batch           : dict from collate_multimodal with corruption metadata
        lambda_reg      : weight of regularization loss

    Returns:
        reg_loss: scalar tensor
    """
    B = img_reliability.shape[0]
    device = img_reliability.device

    # Build target reliability scores from corruption metadata
    img_targets = torch.ones(B, 1, device=device)
    txt_targets = torch.ones(B, 1, device=device)

    for i in range(B):
        sev = batch["corruption_severity"][i]

        # Image target
        if batch["image_missing"][i]:
            img_targets[i] = 0.0        # completely unreliable
        elif batch["image_corrupted"][i]:
            img_targets[i] = 1.0 - sev  # linearly decreasing with severity

        # Text target
        if batch["text_missing"][i]:
            txt_targets[i] = 0.0
        elif batch["text_corrupted"][i]:
            txt_targets[i] = 1.0 - sev

    # MSE between predicted and target reliability scores
    img_reg = F.mse_loss(img_reliability, img_targets)
    txt_reg = F.mse_loss(txt_reliability, txt_targets)

    return lambda_reg * (img_reg + txt_reg)


# ── Test ──────────────────────────────────────────────────────────────────────
print("Testing reliability_regularization_loss...")

# Simulate a batch with mixed corruption states
test_batch_meta = {
    "image_corrupted"    : [False, True,  False, True ],
    "text_corrupted"     : [False, False, True,  True ],
    "image_missing"      : [False, False, False, False],
    "text_missing"       : [False, False, False, False],
    "corruption_severity": [0.0,   0.6,   0.4,   0.8  ],
}

pred_img_rel = torch.tensor([[0.9], [0.8], [0.7], [0.3]])
pred_txt_rel = torch.tensor([[0.8], [0.9], [0.2], [0.1]])

reg_loss = reliability_regularization_loss(
    pred_img_rel, pred_txt_rel, test_batch_meta, lambda_reg=0.1)

print(f"Regularization loss: {reg_loss.item():.4f}")
print()
print("Sample-level analysis:")
print(f"  Sample 0: clean  -> img_target=1.0, txt_target=1.0")
print(f"  Sample 1: img corrupted (sev=0.6) -> img_target={1-0.6:.1f}, txt_target=1.0")
print(f"  Sample 2: txt corrupted (sev=0.4) -> img_target=1.0, txt_target={1-0.4:.1f}")
print(f"  Sample 3: both corrupted (sev=0.8) -> img_target={1-0.8:.1f}, txt_target={1-0.8:.1f}")


## 7. The Full ARAF Model

Now we assemble all components into the complete ARAF model.
The forward pass has four stages:

1. **Encode**: run image through ResNet, text through BERT
2. **Estimate**: run each feature vector through its reliability estimator
3. **Fuse**: weighted combination using reliability scores
4. **Classify**: MLP on the fused representation

The only difference from NaiveFusion is steps 2 and 3.
Everything else is identical — same encoders, same classifier architecture.


In [ ]:
class ARAFModel(nn.Module):
    """
    Adaptive Reliability-Aware Fusion model.

    Extends NaiveFusion with:
    - Per-modality reliability estimation
    - Reliability-weighted adaptive fusion
    - Auxiliary regularization loss for training the estimator

    Architecture:
        Image [3,224,224] -> ResNet-50 -> [2048] -> ReliabilityEstimator -> r_img
                                       -> AdaptiveFusion (weighted by r_img, r_txt)
        Text  [32]        -> BERT     -> [768]  -> ReliabilityEstimator -> r_txt
                                       -> [512] -> MLP -> [num_classes]

    Args:
        num_classes      : output classes (3129 for VQA v2)
        fusion_dim       : common projection dimension for fusion (default 512)
        frozen_encoders  : whether to freeze ResNet and BERT weights
        normalize_weights: whether to normalize reliability scores to sum to 1
        lambda_reg       : weight of reliability regularization loss
    """
    def __init__(
        self,
        num_classes      : int   = 3129,
        fusion_dim       : int   = 512,
        frozen_encoders  : bool  = True,
        normalize_weights: bool  = True,
        lambda_reg       : float = 0.1,
    ):
        super().__init__()
        self.lambda_reg = lambda_reg
        self.name       = "ARAF"

        # ── Encoders (same as baselines) ──────────────────────────────────────
        self.image_encoder = ImageEncoder(frozen=frozen_encoders)
        self.text_encoder  = TextEncoder(frozen=frozen_encoders)

        img_dim = self.image_encoder.feature_dim  # 2048
        txt_dim = self.text_encoder.feature_dim   # 768

        # ── Reliability estimators (new) ──────────────────────────────────────
        self.img_reliability = ReliabilityEstimator(feature_dim=img_dim)
        self.txt_reliability = ReliabilityEstimator(feature_dim=txt_dim)

        # ── Adaptive fusion (new) ─────────────────────────────────────────────
        self.fusion = AdaptiveFusion(
            img_dim          = img_dim,
            txt_dim          = txt_dim,
            fusion_dim       = fusion_dim,
            normalize_weights= normalize_weights,
        )

        # ── Classification head ───────────────────────────────────────────────
        self.classifier = ClassificationHead(
            input_dim  = fusion_dim,
            hidden_dim = 512,
            num_classes= num_classes,
        )

    def forward(self, batch: Dict, return_reliability: bool = False) -> Dict:
        """
        Full ARAF forward pass.

        Args:
            batch             : dict from collate_multimodal
            return_reliability: if True, include reliability scores in output
                                (useful for visualization and debugging)
        Returns:
            dict with:
                logits          : [B, num_classes]
                img_reliability : [B, 1]
                txt_reliability : [B, 1]
                img_weight      : [B, 1]  (normalized)
                txt_weight      : [B, 1]  (normalized)
                fused_features  : [B, fusion_dim]
        """
        images   = batch["image"].to(DEVICE)
        text_ids = batch["text_ids"].to(DEVICE)
        attn     = batch["attention_mask"].to(DEVICE)

        # ── Stage 1: Encode ───────────────────────────────────────────────────
        img_feat = self.image_encoder(images)          # [B, 2048]
        txt_feat = self.text_encoder(text_ids, attn)   # [B, 768]

        # ── Stage 2: Estimate reliability ─────────────────────────────────────
        img_rel = self.img_reliability(img_feat)       # [B, 1]
        txt_rel = self.txt_reliability(txt_feat)       # [B, 1]

        # ── Stage 3: Adaptive fusion ──────────────────────────────────────────
        fusion_out = self.fusion(img_feat, txt_feat, img_rel, txt_rel)

        # ── Stage 4: Classify ─────────────────────────────────────────────────
        logits = self.classifier(fusion_out["fused"])  # [B, num_classes]

        result = {
            "logits"         : logits,
            "img_reliability": img_rel,
            "txt_reliability": txt_rel,
            "img_weight"     : fusion_out["img_weight"],
            "txt_weight"     : fusion_out["txt_weight"],
            "fused_features" : fusion_out["fused"],
            "img_features"   : img_feat,
            "txt_features"   : txt_feat,
        }
        return result

    def compute_loss(self, batch: Dict, output: Dict) -> Dict:
        """
        Compute total loss = task loss + reliability regularization loss.

        Args:
            batch : dict from collate_multimodal (has corruption metadata)
            output: dict from forward()
        Returns:
            dict with:
                total_loss    : scalar (what we call .backward() on)
                task_loss     : scalar (VQA BCE loss)
                reg_loss      : scalar (reliability regularization)
        """
        labels    = batch["label"].to(DEVICE)
        task_loss = vqa_loss(output["logits"], labels)
        reg_loss  = reliability_regularization_loss(
            output["img_reliability"],
            output["txt_reliability"],
            batch,
            self.lambda_reg,
        )
        total_loss = task_loss + reg_loss
        return {
            "total_loss": total_loss,
            "task_loss" : task_loss,
            "reg_loss"  : reg_loss,
        }


# ── Build and test ────────────────────────────────────────────────────────────
print("Building ARAF model...")
araf_model = ARAFModel(
    num_classes       = NUM_CLASSES,
    fusion_dim        = 512,
    frozen_encoders   = True,
    normalize_weights = True,
    lambda_reg        = 0.1,
).to(DEVICE)

# Count parameters
total_p     = sum(p.numel() for p in araf_model.parameters())
trainable_p = sum(p.numel() for p in araf_model.parameters() if p.requires_grad)
frozen_p    = total_p - trainable_p

print(f"Total params     : {total_p:,}")
print(f"Frozen params    : {frozen_p:,}  (ResNet + BERT encoders)")
print(f"Trainable params : {trainable_p:,}  (reliability estimators + fusion + classifier)")


## 8. Test the full forward pass

We run a small batch through ARAF and verify every output tensor
has the correct shape. We also compute the loss to make sure
the training pipeline will work end-to-end.


In [ ]:
# ── Build test batch ──────────────────────────────────────────────────────────
corruption_module = CorruptionModule(
    p_corrupt_image=0.5, p_corrupt_text=0.5,
    p_missing_image=0.1, p_missing_text=0.1,
)

# Apply corruption to 4 samples
corrupted_samples = [corruption_module(samples[i]) for i in range(4)]

def make_batch(sample_list):
    return {
        "image"             : torch.stack([s.image for s in sample_list]),
        "text_ids"          : torch.stack([s.text_ids for s in sample_list]),
        "attention_mask"    : torch.stack([s.attention_mask for s in sample_list]),
        "label"             : torch.stack([s.label for s in sample_list]),
        "image_corrupted"   : [s.image_corrupted for s in sample_list],
        "text_corrupted"    : [s.text_corrupted for s in sample_list],
        "image_missing"     : [s.image_missing for s in sample_list],
        "text_missing"      : [s.text_missing for s in sample_list],
        "corruption_severity": [s.corruption_severity for s in sample_list],
        "raw_text"          : [s.raw_text for s in sample_list],
        "sample_id"         : [s.sample_id for s in sample_list],
    }

test_batch = make_batch(corrupted_samples)

# ── Forward pass ──────────────────────────────────────────────────────────────
araf_model.eval()
with torch.no_grad():
    output = araf_model(test_batch)
    losses = araf_model.compute_loss(test_batch, output)

print("ARAF Forward Pass Results:")
print(f"  logits shape          : {output['logits'].shape}")
print(f"  img_reliability shape : {output['img_reliability'].shape}")
print(f"  txt_reliability shape : {output['txt_reliability'].shape}")
print(f"  fused_features shape  : {output['fused_features'].shape}")
print()
print("Reliability scores (random init, ~0.5 expected):")
for i in range(4):
    img_r = output['img_reliability'][i].item()
    txt_r = output['txt_reliability'][i].item()
    img_w = output['img_weight'][i].item()
    txt_w = output['txt_weight'][i].item()
    img_c = "corrupted" if test_batch['image_corrupted'][i] else "clean"
    txt_c = "corrupted" if test_batch['text_corrupted'][i] else "clean"
    print(f"  Sample {i}: img={img_c} rel={img_r:.3f} w={img_w:.3f} | "
          f"txt={txt_c} rel={txt_r:.3f} w={txt_w:.3f}")
print()
print("Loss breakdown:")
print(f"  task_loss  : {losses['task_loss'].item():.4f}")
print(f"  reg_loss   : {losses['reg_loss'].item():.4f}")
print(f"  total_loss : {losses['total_loss'].item():.4f}")
print()
print("VQA accuracy (random weights): "
      f"{vqa_accuracy(output['logits'], test_batch['label'].to(DEVICE)):.4f}")


## 9. Visualize reliability scores

This is one of the most important visualizations in the project.
After training, we expect the reliability estimator to output:
- High scores for clean inputs
- Low scores for corrupted or missing inputs

Right now with random weights the scores are ~0.5 regardless.
After training (Notebook 05), run this cell again to see the
estimator working correctly. This figure is also excellent for
your paper — it directly demonstrates what ARAF learned.


In [ ]:
def visualize_reliability_scores(model, base_samples, n=8):
    """
    Show reliability scores for clean vs corrupted samples side by side.
    Each row is one sample. Columns show different corruption scenarios.
    """
    model.eval()
    scenarios = {
        "Clean"        : CorruptionModule(p_corrupt_image=0.0, p_corrupt_text=0.0,
                                          p_missing_image=0.0, p_missing_text=0.0),
        "Img noise s3" : CorruptionModule(p_corrupt_image=1.0, p_corrupt_text=0.0,
                                          p_missing_image=0.0, p_missing_text=0.0,
                                          severity=3, image_corruptions=["gaussian_noise"]),
        "Img noise s5" : CorruptionModule(p_corrupt_image=1.0, p_corrupt_text=0.0,
                                          p_missing_image=0.0, p_missing_text=0.0,
                                          severity=5, image_corruptions=["gaussian_noise"]),
        "Img missing"  : CorruptionModule(p_corrupt_image=0.0, p_corrupt_text=0.0,
                                          p_missing_image=1.0, p_missing_text=0.0),
        "Txt dropout s3": CorruptionModule(p_corrupt_image=0.0, p_corrupt_text=1.0,
                                           p_missing_image=0.0, p_missing_text=0.0,
                                           severity=3, text_corruptions=["token_dropout"]),
        "Txt missing"  : CorruptionModule(p_corrupt_image=0.0, p_corrupt_text=0.0,
                                          p_missing_image=0.0, p_missing_text=1.0),
    }

    scenario_names = list(scenarios.keys())
    n_scenarios    = len(scenario_names)
    n_samples      = min(n, len(base_samples))

    img_scores_all = np.zeros((n_samples, n_scenarios))
    txt_scores_all = np.zeros((n_samples, n_scenarios))

    with torch.no_grad():
        for j, (name, corr) in enumerate(scenarios.items()):
            for i in range(n_samples):
                s     = corr(base_samples[i])
                batch = make_batch([s])
                out   = model(batch)
                img_scores_all[i, j] = out["img_reliability"].item()
                txt_scores_all[i, j] = out["txt_reliability"].item()

    # ── Plot heatmaps ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, scores, title in zip(
        axes,
        [img_scores_all, txt_scores_all],
        ["Image reliability scores", "Text reliability scores"]
    ):
        im = ax.imshow(scores, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
        ax.set_xticks(range(n_scenarios))
        ax.set_xticklabels(scenario_names, rotation=30, ha="right", fontsize=9)
        ax.set_yticks(range(n_samples))
        ax.set_yticklabels([f"Sample {i}" for i in range(n_samples)], fontsize=8)
        ax.set_title(title, fontsize=11)

        # Annotate cells with score values
        for i in range(n_samples):
            for j in range(n_scenarios):
                ax.text(j, i, f"{scores[i,j]:.2f}",
                        ha="center", va="center", fontsize=7,
                        color="black")

        plt.colorbar(im, ax=ax, label="Reliability (0=bad, 1=good)")

    fig.suptitle(
        "Reliability scores across corruption scenarios" + "
"
        "(~0.5 with random weights -- will show clear patterns after training)",
        fontsize=12
    )
    plt.tight_layout()
    plt.savefig("reliability_scores.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: reliability_scores.png")
    print()
    print("What to look for after training:")
    print("  Column 'Clean'       -> all cells should be GREEN (high scores)")
    print("  Column 'Img missing' -> image row should be RED (low), text GREEN")
    print("  Column 'Txt missing' -> text row should be RED (low), image GREEN")
    print("  Column 'Img noise s5'-> image score should drop, text stay high")

visualize_reliability_scores(araf_model, samples, n=8)


## 10. ARAF vs Naive Fusion — parameter comparison

Before training, let's make sure we understand exactly what is
different between ARAF and NaiveFusion in terms of architecture
and parameter count. This is important for your paper's
"model complexity" section.


In [ ]:
def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# Build NaiveFusion for comparison
naive_model = baselines.NaiveFusionModel(num_classes=NUM_CLASSES).to(DEVICE)

naive_total, naive_train   = count_params(naive_model)
araf_total,  araf_train    = count_params(araf_model)

print("=" * 55)
print(f"{'Model':<20} {'Total':>12} {'Trainable':>12}")
print("=" * 55)
print(f"{'NaiveFusion':<20} {naive_total:>12,} {naive_train:>12,}")
print(f"{'ARAF':<20} {araf_total:>12,} {araf_train:>12,}")
print("=" * 55)
overhead = araf_train - naive_train
print(f"ARAF overhead    : {overhead:>12,} extra trainable params")
print(f"Overhead %       : {overhead/naive_train*100:.2f}% of NaiveFusion trainable")
print()
print("Breakdown of ARAF-specific components:")
rel_params = sum(p.numel() for p in araf_model.img_reliability.parameters())
rel_params+= sum(p.numel() for p in araf_model.txt_reliability.parameters())
fus_params = sum(p.numel() for p in araf_model.fusion.parameters())
print(f"  Reliability estimators : {rel_params:,}")
print(f"  Adaptive fusion        : {fus_params:,}")
print(f"  Total ARAF additions   : {rel_params + fus_params:,}")
print()
print("Key point for your paper:")
print("  ARAF adds minimal parameters over NaiveFusion.")
print("  The performance gain comes from the architecture design,")
print("  not from having more parameters.")


## 11. Save ARAF model to `models/araf.py`


In [ ]:
import os
os.makedirs("models", exist_ok=True)

araf_lines = [
    "# models/araf.py",
    "# ARAF model for the ARAF project.",
    "# Validated in Notebook 04.",
    "from typing import Optional, Dict",
    "import torch",
    "import torch.nn as nn",
    "import torch.nn.functional as F",
    "import importlib.util, os",
    "",
    "def _load_baselines():",
    "    path = os.path.join(os.path.dirname(__file__), 'baselines.py')",
    "    spec = importlib.util.spec_from_file_location('baselines', path)",
    "    mod  = importlib.util.module_from_spec(spec)",
    "    spec.loader.exec_module(mod)",
    "    return mod",
    "",
    "class ReliabilityEstimator(nn.Module):",
    "    def __init__(self, feature_dim, hidden_dim=256, dropout=0.2):",
    "        super().__init__()",
    "        self.estimator = nn.Sequential(",
    "            nn.Linear(feature_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),",
    "            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),",
    "            nn.Linear(hidden_dim // 2, 1), nn.Sigmoid(),",
    "        )",
    "    def forward(self, x): return self.estimator(x)",
    "",
    "class AdaptiveFusion(nn.Module):",
    "    def __init__(self, img_dim=2048, txt_dim=768, fusion_dim=512, normalize_weights=True):",
    "        super().__init__()",
    "        self.fusion_dim = fusion_dim",
    "        self.normalize_weights = normalize_weights",
    "        self.img_proj = nn.Sequential(nn.Linear(img_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.ReLU())",
    "        self.txt_proj = nn.Sequential(nn.Linear(txt_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.ReLU())",
    "    def forward(self, img_f, txt_f, img_rel, txt_rel):",
    "        img_p = self.img_proj(img_f)",
    "        txt_p = self.txt_proj(txt_f)",
    "        if self.normalize_weights:",
    "            eps = 1e-8",
    "            total = img_rel + txt_rel + eps",
    "            img_w, txt_w = img_rel / total, txt_rel / total",
    "        else:",
    "            img_w, txt_w = img_rel, txt_rel",
    "        fused = img_w * img_p + txt_w * txt_p",
    "        return {'fused': fused, 'img_weight': img_w, 'txt_weight': txt_w,",
    "                'img_proj': img_p, 'txt_proj': txt_p}",
    "",
    "def reliability_regularization_loss(img_rel, txt_rel, batch, lambda_reg=0.1):",
    "    B, device = img_rel.shape[0], img_rel.device",
    "    img_t = torch.ones(B, 1, device=device)",
    "    txt_t = torch.ones(B, 1, device=device)",
    "    for i in range(B):",
    "        sev = batch['corruption_severity'][i]",
    "        if batch['image_missing'][i]:   img_t[i] = 0.0",
    "        elif batch['image_corrupted'][i]: img_t[i] = 1.0 - sev",
    "        if batch['text_missing'][i]:    txt_t[i] = 0.0",
    "        elif batch['text_corrupted'][i]:  txt_t[i] = 1.0 - sev",
    "    return lambda_reg * (F.mse_loss(img_rel, img_t) + F.mse_loss(txt_rel, txt_t))",
    "",
    "class ARAFModel(nn.Module):",
    "    def __init__(self, num_classes=3129, fusion_dim=512,",
    "                 frozen_encoders=True, normalize_weights=True, lambda_reg=0.1):",
    "        super().__init__()",
    "        self.lambda_reg = lambda_reg",
    "        self.name = 'ARAF'",
    "        bl = _load_baselines()",
    "        self.image_encoder = bl.ImageEncoder(frozen=frozen_encoders)",
    "        self.text_encoder  = bl.TextEncoder(frozen=frozen_encoders)",
    "        self.vqa_loss      = bl.vqa_loss",
    "        self.vqa_accuracy  = bl.vqa_accuracy",
    "        img_dim = self.image_encoder.feature_dim",
    "        txt_dim = self.text_encoder.feature_dim",
    "        self.img_reliability = ReliabilityEstimator(img_dim)",
    "        self.txt_reliability = ReliabilityEstimator(txt_dim)",
    "        self.fusion    = AdaptiveFusion(img_dim, txt_dim, fusion_dim, normalize_weights)",
    "        self.classifier = bl.ClassificationHead(fusion_dim, hidden_dim=512, num_classes=num_classes)",
    "    def forward(self, batch, device='cpu'):",
    "        images   = batch['image'].to(device)",
    "        text_ids = batch['text_ids'].to(device)",
    "        attn     = batch['attention_mask'].to(device)",
    "        img_feat = self.image_encoder(images)",
    "        txt_feat = self.text_encoder(text_ids, attn)",
    "        img_rel  = self.img_reliability(img_feat)",
    "        txt_rel  = self.txt_reliability(txt_feat)",
    "        fout     = self.fusion(img_feat, txt_feat, img_rel, txt_rel)",
    "        logits   = self.classifier(fout['fused'])",
    "        return {'logits': logits, 'img_reliability': img_rel,",
    "                'txt_reliability': txt_rel, 'img_weight': fout['img_weight'],",
    "                'txt_weight': fout['txt_weight'], 'fused_features': fout['fused'],",
    "                'img_features': img_feat, 'txt_features': txt_feat}",
    "    def compute_loss(self, batch, output, device='cpu'):",
    "        labels     = batch['label'].to(device)",
    "        task_loss  = self.vqa_loss(output['logits'], labels)",
    "        reg_loss   = reliability_regularization_loss(",
    "            output['img_reliability'], output['txt_reliability'],",
    "            batch, self.lambda_reg)",
    "        return {'total_loss': task_loss + reg_loss,",
    "                'task_loss': task_loss, 'reg_loss': reg_loss}",
]

with open("models/araf.py", "w", encoding="utf-8") as f:
    f.write("\n".join(araf_lines))

print("Saved: models/araf.py")

spec3 = importlib.util.spec_from_file_location("araf", "models/araf.py")
araf_mod = importlib.util.module_from_spec(spec3)
spec3.loader.exec_module(araf_mod)
print("Import verified. ARAFModel ready.")


## 12. Summary and what's next

### What we built

| Component | Role |
|---|---|
| `ReliabilityEstimator` | Small MLP: features -> scalar score in [0,1] |
| `AdaptiveFusion` | Projects modalities to common dim, weighted sum |
| `reliability_regularization_loss` | Trains estimator using corruption metadata |
| `ARAFModel` | Full model: encode -> estimate -> fuse -> classify |
| `compute_loss` | task BCE loss + regularization loss |

### The key insight in one sentence

ARAF does not add much complexity over NaiveFusion — it just asks
"how reliable is each modality?" before combining them, and uses
the answer to weight the combination.

### What Notebook 05 covers

**Training** — putting everything together into a training loop that:
- Loads batches from VQA v2
- Applies random corruption via CorruptionModule
- Runs the forward pass through ARAF (and baselines for comparison)
- Computes task loss + reliability regularization loss
- Updates weights via Adam optimizer
- Evaluates on clean and corrupted validation sets after each epoch
- Saves the best checkpoint

After Notebook 05 you will have a trained ARAF model and the
robustness curves will show real, meaningful differences between
ARAF and the baselines.

### Your project folder now

```
your_project/
├── corruption/corruption_module.py
├── models/
│   ├── __init__.py
│   ├── baselines.py
│   └── araf.py          <- added this notebook
├── *.png
```
